# GameTheory-13d : Optimistic Counterfactual Regret Minimization — jumeau C#

**Navigation** : [<< 13d-Optimistic-CFR-Python](GameTheory-13d-Optimistic-CFR-Python.ipynb) | [Index](../README.md) | [13e-Revision-Croyances-Adversaire >>](GameTheory-13e-Revision-Croyances-Adversaire-Python.ipynb)

**Twin C# (.NET Interactive)** de [GameTheory-13d-Optimistic-CFR-Python](GameTheory-13d-Optimistic-CFR-Python.ipynb) — parité .NET ⇄ Python.

> **Notebook compagnon** du GT-13, isolant la famille **optimiste** du CFR : l'**OFTRL** (*Stable-Predictive Optimistic Counterfactual Regret Minimization*, Farina, Kroer, Brown, Sandholm, ICML 2019, PMLR 97) et le **DCFR** (Brown & Sandholm 2019). Le jumeau C# traduit le cœur pédagogique — les solveurs **CFR / CFR+ / OFTRL / DCFR** en **BCL .NET 9 natif** (0 NuGet pour le socle) — **et** appelle **réellement OpenSpiel via Python.NET** pour l'exploitabilité réelle (meilleure réponse) : le pont .NET → CPython → `pyspiel` → moteur OpenSpiel C++ (§5, pont déjà démontré dans la série).

## Objectifs d'apprentissage

- Comprendre la **garantie théorique** $O(T^{-3/4})$ de l'OFTRL stable-prédictif, contre $O(T^{-1/2})$ du CFR/CFR+ — et ce que la régression empirique *réellement* observe sur Kuhn poker.
- Distinguer **mises à jour simultanées** (toutes les cartes / tour) et **mises à jour alternées** (une carte / tour).
- Régler empiriquement $(\kappa, \alpha, \beta)$ de l'OFTRL sur la profondeur de la régression.
- Lire honnêtement le banc : sur Kuhn poker, le **DCFR arrive dernier** ici — l'OFTRL n'est pas une amélioration universelle, c'est un pari sur la profondeur.

> **Parité avec le jumeau Python.** Le jumeau Python charge son socle (CFR / CFR+ / MCCFR) par `importlib` depuis le carnet GT-13 ; le jumeau C# **définit son socle en place** (une session .NET Interactive est auto-suffisante — pas d'équivalent `importlib`). L'exploitabilité, que le Python délègue à OpenSpiel, est mesurée ici **par un appel réel OpenSpiel via Python.NET** (§5).

***

*Part of #26082 · See #26902 / #26070 · Suivi Marathon II hors flotte #17713*

## Préambule : le socle, défini en place

Le jumeau Python charge ses classes de base (CFR / CFR+ / MCCFR) par `importlib.util.spec_from_file_location` depuis le carnet GT-13 principal. En .NET Interactive, chaque carnet est une **session de kernel auto-suffisante** : il n'existe pas d'`importlib` équivalent, le socle est donc **défini ici**. Ce sont les mêmes classes que le jumeau C# du GT-13 :

| Classe | Rôle |
|---|---|
| `Kuhn` | modèle du jeu — payoffs, infosets, joueur courant, les 12 infosets |
| `CFRSolver` | récursion contrefactuelle vanilla (base) |
| `CFRPlusSolver` | variante à regrets écrêtés (Tammelin 2014) |

Le présent carnet n'écrit que **deux** nouveaux solveurs — `OFTRLSolver` (§3) et `DCFRDiscountedSolver` (§4) — puis le banc de mesure OpenSpiel (§5).

In [1]:
using System.Linq;
using System.Collections.Generic;

// Kuhn Poker : 3 cartes (J=0, Q=1, K=2), 2 actions (Pass=0, Bet=1).
public static class Kuhn
{
    public const int PASS = 0;
    public const int BET = 1;
    public const int NUM_ACTIONS = 2;
    public static readonly string[] CardName = { "J", "Q", "K" };

    public static bool IsTerminal(string history)
        => history == "pp" || history == "pbp" || history == "pbb"
           || history == "bp" || history == "bb";

    public static double GetPayoff(string history, int[] cards)
    {
        bool p1Higher = cards[0] > cards[1];
        return history switch
        {
            "pp"  => p1Higher ? 1 : -1,
            "pbp" => -1,
            "pbb" => p1Higher ? 2 : -2,
            "bp"  => 1,
            "bb"  => p1Higher ? 2 : -2,
            _ => 0
        };
    }

    public static string GetInfoSet(string history, int card) => CardName[card] + history;
    public static int GetCurrentPlayer(string history) => history.Length % 2;

    // Les 12 infosets atteignables (3 cartes x 4 positions d'historique).
    public static string[] AllInfoSets()
    {
        var list = new List<string>();
        foreach (var card in new[] { 0, 1, 2 })
            foreach (var h in new[] { "", "b", "p", "pb" })
                list.Add(CardName[card] + h);
        return list.ToArray();
    }
}

// Solveur CFR vanilla pour Kuhn Poker : regret_sum et strategy_sum par infoset.
public class CFRSolver
{
    public Dictionary<string, double[]> RegretSum = new();
    public Dictionary<string, double[]> StrategySum = new();
    public int Iterations = 0;

    protected virtual double[] GetRegret(string infoset)
    {
        if (!RegretSum.ContainsKey(infoset)) RegretSum[infoset] = new double[Kuhn.NUM_ACTIONS];
        return RegretSum[infoset];
    }

    public virtual double[] GetStrategy(string infoset)
    {
        var regrets = GetRegret(infoset);
        var s = new double[Kuhn.NUM_ACTIONS];
        double norm = 0;
        for (int a = 0; a < Kuhn.NUM_ACTIONS; a++) { s[a] = Math.Max(0, regrets[a]); norm += s[a]; }
        if (norm > 0)
            for (int a = 0; a < Kuhn.NUM_ACTIONS; a++) s[a] /= norm;
        else
            for (int a = 0; a < Kuhn.NUM_ACTIONS; a++) s[a] = 1.0 / Kuhn.NUM_ACTIONS;
        return s;
    }

    public double[] GetAverageStrategy(string infoset)
    {
        if (!StrategySum.ContainsKey(infoset))
            return Enumerable.Repeat(1.0 / Kuhn.NUM_ACTIONS, Kuhn.NUM_ACTIONS).ToArray();
        var ss = StrategySum[infoset];
        double norm = ss.Sum();
        var s = new double[Kuhn.NUM_ACTIONS];
        if (norm > 0)
            for (int a = 0; a < Kuhn.NUM_ACTIONS; a++) s[a] = ss[a] / norm;
        else
            for (int a = 0; a < Kuhn.NUM_ACTIONS; a++) s[a] = 1.0 / Kuhn.NUM_ACTIONS;
        return s;
    }

    public virtual double[] Cfr(string history, int[] cards, double[] reach, int depth = 0)
    {
        if (Kuhn.IsTerminal(history))
        {
            double p = Kuhn.GetPayoff(history, cards);
            return new double[] { p, -p };
        }
        int player = Kuhn.GetCurrentPlayer(history);
        int opponent = 1 - player;
        string infoset = Kuhn.GetInfoSet(history, cards[player]);
        var strategy = GetStrategy(infoset);

        var childUtil = new double[Kuhn.NUM_ACTIONS][];
        double[] nodeUtil = { 0, 0 };
        for (int a = 0; a < Kuhn.NUM_ACTIONS; a++)
        {
            char ac = a == Kuhn.PASS ? 'p' : 'b';
            var newReach = (double[])reach.Clone();
            newReach[player] *= strategy[a];
            childUtil[a] = Cfr(history + ac, cards, newReach, depth);
            nodeUtil[0] += strategy[a] * childUtil[a][0];
            nodeUtil[1] += strategy[a] * childUtil[a][1];
        }

        double cfReach = reach[opponent];
        var rs = GetRegret(infoset);
        for (int a = 0; a < Kuhn.NUM_ACTIONS; a++)
            rs[a] += cfReach * (childUtil[a][player] - nodeUtil[player]);

        if (!StrategySum.ContainsKey(infoset)) StrategySum[infoset] = new double[Kuhn.NUM_ACTIONS];
        var ss = StrategySum[infoset];
        for (int a = 0; a < Kuhn.NUM_ACTIONS; a++) ss[a] += reach[player] * strategy[a];
        return nodeUtil;
    }

    // Les 6 donnees (J1, J2) avec deux cartes distinctes.
    public static readonly int[][] CardPermutations = {
        new[]{0,1}, new[]{0,2}, new[]{1,0}, new[]{1,2}, new[]{2,0}, new[]{2,1}
    };
}

// CFR+ : variant a regrets ecletes (Tammelin 2014).
public class CFRPlusSolver : CFRSolver
{
    public override double[] Cfr(string history, int[] cards, double[] reach, int depth = 0)
    {
        if (Kuhn.IsTerminal(history))
        {
            double p = Kuhn.GetPayoff(history, cards);
            return new double[] { p, -p };
        }
        int player = Kuhn.GetCurrentPlayer(history);
        int opponent = 1 - player;
        string infoset = Kuhn.GetInfoSet(history, cards[player]);
        var strategy = GetStrategy(infoset);

        var childUtil = new double[Kuhn.NUM_ACTIONS][];
        double[] nodeUtil = { 0, 0 };
        for (int a = 0; a < Kuhn.NUM_ACTIONS; a++)
        {
            char ac = a == Kuhn.PASS ? 'p' : 'b';
            var newReach = (double[])reach.Clone();
            newReach[player] *= strategy[a];
            childUtil[a] = Cfr(history + ac, cards, newReach, depth);
            nodeUtil[0] += strategy[a] * childUtil[a][0];
            nodeUtil[1] += strategy[a] * childUtil[a][1];
        }

        double cfReach = reach[opponent];
        var rs = GetRegret(infoset);
        for (int a = 0; a < Kuhn.NUM_ACTIONS; a++)
            rs[a] = Math.Max(0, rs[a] + cfReach * (childUtil[a][player] - nodeUtil[player]));

        if (!StrategySum.ContainsKey(infoset)) StrategySum[infoset] = new double[Kuhn.NUM_ACTIONS];
        var ss = StrategySum[infoset];
        for (int a = 0; a < Kuhn.NUM_ACTIONS; a++) ss[a] += reach[player] * strategy[a];
        return nodeUtil;
    }
}

$"Socle charge : Kuhn ({Kuhn.AllInfoSets().Length} infosets), CFRSolver, CFRPlusSolver".Display();
$"Donnees (6 permutations) : {string.Join(", ", CFRSolver.CardPermutations.Select(c => string.Join("-", c)))}".Display();

The below script needs to be able to find the current output cell; this is an easy method to get it.

Socle charge : Kuhn (12 infosets), CFRSolver, CFRPlusSolver

Donnees (6 permutations) : 0-1, 0-2, 1-0, 1-2, 2-0, 2-1

## 1. OFTRL — *Stable-Predictive Optimistic Counterfactual Regret Minimization*

La **récurrence standard** du CFR met à jour

$$
R_{t+1}(i) \;=\; R_t(i) + \text{cfReach}_t \cdot \text{regret}_t(i)
$$

puis joue `argmin` sur les regrets (écrêtés à 0 pour CFR+). L'**OFTRL** ajoute un terme de **prédiction** $m_t(i)$ du regret *à venir*, injecté dans le calcul de la stratégie :

$$
m_{t+1}(i) \;=\; m_t(i) + \frac{\kappa}{T + \beta} \cdot \frac{1}{1 + \alpha\, d} \cdot \max\!\big(R_t(i),\, 0\big)
$$

où $d$ est la **profondeur** de la régression et $T = \max(\text{it}, 1)$. Les trois hyperparamètres :

| Paramètre | Rôle | Effet |
|---|---|---|
| $\kappa$ | gain de la prédiction | plus grand → plus optimiste |
| $\alpha$ | décroissance en profondeur | plus grand → la prédiction s'éteint vite avec la profondeur |
| $\beta$ | lissage du pas $1/(T+\beta)$ | plus grand → pas plus petit au démarrage |

**Garantie théorique** (théorème 1, Farina et al. 2019) : avec la bonne calibration, l'exploitabilité moyenne du CFR optimiste décroît en $O(T^{-3/4})$ contre $O(T^{-1/2})$ du CFR classique. L'OFTRL est le CFR optimiste le plus simple à *prouver* ; il n'est pas le plus fort en pratique (cf. le banc §5).

> **C.1** — aucun `throw` / `assert False` / `1/0` : le carnet s'exécute de bout en bout.

In [2]:
// OFTRLSolver : pas optimiste stable-predictif (Farina, Kroer, Brown, Sandholm, ICML 2019).
// Injecte une prediction m_{t+1} du regret, calibree par profondeur (kappa) avec
// amortissement (alpha, beta). La strategie courante = argmin sur (regret + prediction).
public class OFTRLSolver : CFRSolver
{
    public Dictionary<string, double[]> PredictSum = new();
    double _kappa, _alpha, _beta;

    public OFTRLSolver(double kappa = 1.0, double alpha = 0.5, double beta = 1.0)
    {
        _kappa = kappa; _alpha = alpha; _beta = beta;
    }

    // Surcharge : la strategie combine regrets et prediction.
    public override double[] GetStrategy(string infoset)
    {
        if (!RegretSum.ContainsKey(infoset)) RegretSum[infoset] = new double[Kuhn.NUM_ACTIONS];
        if (!PredictSum.ContainsKey(infoset)) PredictSum[infoset] = new double[Kuhn.NUM_ACTIONS];
        var r = RegretSum[infoset]; var p = PredictSum[infoset];
        var s = new double[Kuhn.NUM_ACTIONS];
        double norm = 0;
        for (int a = 0; a < Kuhn.NUM_ACTIONS; a++) { s[a] = Math.Max(0, r[a] + p[a]); norm += s[a]; }
        if (norm > 0)
            for (int a = 0; a < Kuhn.NUM_ACTIONS; a++) s[a] /= norm;
        else
            for (int a = 0; a < Kuhn.NUM_ACTIONS; a++) s[a] = 1.0 / Kuhn.NUM_ACTIONS;
        return s;
    }

    public override double[] Cfr(string history, int[] cards, double[] reach, int depth = 0)
    {
        if (Kuhn.IsTerminal(history))
        {
            double p = Kuhn.GetPayoff(history, cards);
            return new double[] { p, -p };
        }
        int player = Kuhn.GetCurrentPlayer(history);
        int opponent = 1 - player;
        string infoset = Kuhn.GetInfoSet(history, cards[player]);
        var strategy = GetStrategy(infoset);

        var childUtil = new double[Kuhn.NUM_ACTIONS][];
        double nodeValue = 0.0;
        double[] actionUtils = new double[Kuhn.NUM_ACTIONS];
        for (int a = 0; a < Kuhn.NUM_ACTIONS; a++)
        {
            char ac = a == Kuhn.PASS ? 'p' : 'b';
            var newReach = (double[])reach.Clone();
            for (int k = 0; k < 2; k++) newReach[k] *= strategy[a];
            childUtil[a] = Cfr(history + ac, cards, newReach, depth + 1);
            actionUtils[a] = childUtil[a][player];
            nodeValue += strategy[a] * actionUtils[a];
        }

        // Mise a jour contrefactuelle des regrets.
        var rs = GetRegret(infoset);
        for (int a = 0; a < Kuhn.NUM_ACTIONS; a++)
            rs[a] += reach[opponent] * (actionUtils[a] - nodeValue);

        // PAS OPTIMISTE : m_{t+1} += step * depth_factor * max(regret, 0).
        int T = Math.Max(Iterations, 1);
        double step = _kappa / (T + _beta);
        double depthFactor = 1.0 / (1.0 + _alpha * depth);
        if (!PredictSum.ContainsKey(infoset)) PredictSum[infoset] = new double[Kuhn.NUM_ACTIONS];
        var ps = PredictSum[infoset];
        for (int a = 0; a < Kuhn.NUM_ACTIONS; a++)
            ps[a] += step * depthFactor * Math.Max(rs[a], 0);

        // Accumulation de la strategie courante.
        if (!StrategySum.ContainsKey(infoset)) StrategySum[infoset] = new double[Kuhn.NUM_ACTIONS];
        var ss = StrategySum[infoset];
        for (int a = 0; a < Kuhn.NUM_ACTIONS; a++) ss[a] += reach[player] * strategy[a];

        return player == 0 ? new double[] { nodeValue, -nodeValue } : new double[] { -nodeValue, nodeValue };
    }
}

// Demonstration : 500 iterations, CFR vs CFR+ vs OFTRL sur l'infoset "Qp".
var oftrl = new OFTRLSolver(1.0, 0.5, 1.0);
var cfr  = new CFRSolver();
var cfrPlus = new CFRPlusSolver();
Console.WriteLine($"{"iter":>6}  {"CFR avg (Qp)":>16}  {"CFR+ avg (Qp)":>16}  {"OFTRL avg (Qp)":>17}");
foreach (var t in new[] { 0, 9, 49, 99, 199, 299, 399, 499 })
{
    while (cfr.Iterations < t)
    {
        cfr.Iterations++;
        cfr.Cfr("", new[]{0,1}, new double[]{1.0,1.0});
        cfrPlus.Iterations = cfr.Iterations; cfrPlus.Cfr("", new[]{0,1}, new double[]{1.0,1.0});
        oftrl.Iterations = cfr.Iterations;  oftrl.Cfr("", new[]{0,1}, new double[]{1.0,1.0});
    }
    var c  = cfr.GetAverageStrategy("Qp");
    var cp = cfrPlus.GetAverageStrategy("Qp");
    var o  = oftrl.GetAverageStrategy("Qp");
    Console.WriteLine($"{cfr.Iterations,6}  {string.Join(",", c.Select(x=>x.ToString("F4"))),16}  {string.Join(",", cp.Select(x=>x.ToString("F4"))),16}  {string.Join(",", o.Select(x=>x.ToString("F4"))),17}");
}
"(strategie moyenne sur l'infoset 'Qp' = J1, carte Q, face a un check)".Display();

iter  CFR avg (Qp)  CFR+ avg (Qp)  OFTRL avg (Qp)


     0     0,5000,0,5000     0,5000,0,5000      0,5000,0,5000


     9     0,0556,0,9444     0,0556,0,9444      0,0359,0,9641


    49     0,0102,0,9898     0,0102,0,9898      0,0053,0,9947


    99     0,0051,0,9949     0,0051,0,9949      0,0026,0,9974


   199     0,0025,0,9975     0,0025,0,9975      0,0013,0,9987


   299     0,0017,0,9983     0,0017,0,9983      0,0008,0,9992


   399     0,0013,0,9987     0,0013,0,9987      0,0006,0,9994


   499     0,0010,0,9990     0,0010,0,9990      0,0005,0,9995


(strategie moyenne sur l'infoset 'Qp' = J1, carte Q, face a un check)

## 2. DCFR — *Discounted Counterfactual Regret Minimization*

Le **DCFR** (Brown & Sandholm 2019) est l'approche « décroissante » : au lieu d'accumuler les regrets en les laissant dominer, on les **discoute** avec un facteur qui dépend de l'itération courante $t$ :

$$
R_{t+1}(i) \;=\; R_t(i) + \text{cfReach}_t \cdot \max(\text{regret}_t(i), 0) \cdot \frac{t^{\alpha}}{t^{\alpha}+1}
$$

**Forme exécutée ici** (fidèle au jumeau Python) : le facteur de regret $t^{\alpha}/(t^{\alpha}+1)$ porte sur **l'incrément** de regret — il tend vers 1, il ne *décroît pas* la somme accumulée. Le facteur de stratégie $t^{\beta}/(t^{\beta}+1)$ est **neutre par défaut** ($\beta=0$ → facteur 1). Ce n'est **pas** la « DCFR à décroissance de la somme accumulée » (qui diviserait $R_t$ à chaque tour) : c'est une variante « incrément discounté », plus douce, qui reste un CFR classique avec un pas qui sature.

| Paramètre | Valeur par défaut | Rôle |
|---|---|---|
| $\alpha$ | 1.5 | courbe du discount de regret |
| $\beta$ | 0.0 | courbe du discount de stratégie (neutre si 0) |
| $\gamma$ | 2.0 | (réservé, non utilisé dans cette forme) |

In [3]:
// DCFRDiscountedSolver : DCFR (Brown & Sandholm 2019). Forme EXECUTEE ici : le facteur
// de regret t^alpha/(t^alpha+1) porte sur l'INCREMENT de regret (tend vers 1, ne
// decroit pas la somme accumulee) ; le facteur de strategie t^beta/(t^beta+1) est
// neutre par defaut (beta=0). Cf §2 : ce n'est pas la forme "decay de la somme accumulee".
public class DCFRDiscountedSolver : CFRSolver
{
    double _alpha, _beta, _gamma;
    public DCFRDiscountedSolver(double alpha = 1.5, double beta = 0.0, double gamma = 2.0)
    { _alpha = alpha; _beta = beta; _gamma = gamma; }

    public override double[] Cfr(string history, int[] cards, double[] reach, int depth = 0)
    {
        if (Kuhn.IsTerminal(history))
        {
            double p = Kuhn.GetPayoff(history, cards);
            return new double[] { p, -p };
        }
        int player = Kuhn.GetCurrentPlayer(history);
        int opponent = 1 - player;
        string infoset = Kuhn.GetInfoSet(history, cards[player]);
        var strategy = GetStrategy(infoset);

        var childUtil = new double[Kuhn.NUM_ACTIONS][];
        double nodeValue = 0.0;
        double[] actionUtils = new double[Kuhn.NUM_ACTIONS];
        for (int a = 0; a < Kuhn.NUM_ACTIONS; a++)
        {
            char ac = a == Kuhn.PASS ? 'p' : 'b';
            var newReach = (double[])reach.Clone();
            for (int k = 0; k < 2; k++) newReach[k] *= strategy[a];
            childUtil[a] = Cfr(history + ac, cards, newReach, depth + 1);
            actionUtils[a] = childUtil[a][player];
            nodeValue += strategy[a] * actionUtils[a];
        }

        int t = Math.Max(Iterations, 1);
        double discount = Math.Pow(t, _alpha) / (Math.Pow(t, _alpha) + 1);
        double stratDiscount = _beta > 0 ? Math.Pow(t, _beta) / (Math.Pow(t, _beta) + 1) : 1.0;

        var rs = GetRegret(infoset);
        for (int a = 0; a < Kuhn.NUM_ACTIONS; a++)
            rs[a] += reach[opponent] * Math.Max(0, actionUtils[a] - nodeValue) * discount;
        if (!StrategySum.ContainsKey(infoset)) StrategySum[infoset] = new double[Kuhn.NUM_ACTIONS];
        var ss = StrategySum[infoset];
        for (int a = 0; a < Kuhn.NUM_ACTIONS; a++) ss[a] += reach[player] * strategy[a] * stratDiscount;

        return player == 0 ? new double[] { nodeValue, -nodeValue } : new double[] { -nodeValue, nodeValue };
    }
}

// Sanity check : convergence sur l'infoset Qp, solveurs instances NEUFS.
var cfr  = new CFRSolver();
var cfrD = new DCFRDiscountedSolver();
"DCFR sanity check (strategie moyenne 'Qp') :".Display();
foreach (var target in new[] { 10, 100, 500 })
{
    while (cfr.Iterations < target)
    {
        cfr.Iterations++;
        cfr.Cfr("", new[]{0,1}, new double[]{1.0,1.0});
        cfrD.Iterations = cfr.Iterations; cfrD.Cfr("", new[]{0,1}, new double[]{1.0,1.0});
    }
    var c = cfr.GetAverageStrategy("Qp");
    var d = cfrD.GetAverageStrategy("Qp");
    $"  t={target,4}   CFR=[{c[0]:F3}, {c[1]:F3}]   DCFR=[{d[0]:F3}, {d[1]:F3}]".Display();
}

DCFR sanity check (strategie moyenne 'Qp') :

  t=  10   CFR=[0,050, 0,950]   DCFR=[0,034, 0,966]

  t= 100   CFR=[0,005, 0,995]   DCFR=[0,003, 0,997]

  t= 500   CFR=[0,001, 0,999]   DCFR=[0,001, 0,999]

## 3. Banc de mesure — exploitabilité **réelle** via OpenSpiel (Python.NET)

La métrique d'un solveur CFR est son **exploitabilité** : combien un adversaire qui connaît parfaitement sa stratégie peut en gagner (meilleure réponse). Le jumeau Python délègue ce calcul à OpenSpiel ; ici on l'obtient par un **appel réel** du moteur OpenSpiel, depuis C#, via le pont **Python.NET** :

```
.NET Interactive (C#) ──Python.NET──▶ CPython (venv 3.13) ──pyspiel──▶ OpenSpiel C++
```

Le pont est **déjà démontré** dans la série (jumeau C# du GT-13, annexe §7 : CFR 100 iters → exploitabilité 0.008226, verdict `RECOVERABLE-LOCAL`). La cellule qui suit recharge le runtime et expose un helper `OpenSpielBridge.Exploitability(solveur)` qui :

1. traduit chaque infoset C# (`"Qp"`, carte Q=1, historique `p`) en id OpenSpiel (`"1p"` : **chiffre** de la carte + historique) ;
2. assemble la politique en `pyspiel.TabularPolicy(dict)` ;
3. appelle `pyspiel.exploitability(game, policy)`.

> **C.1 / SOTA** — l'exploitabilité est mesurée par l'outil **réel** (OpenSpiel), jamais approxmée par une distance à l'uniforme.

In [12]:
#r "nuget: pythonnet,3.1.0"
#load "../Config/PythonNetLoader.cs"
using System;
using System.Collections.Generic;
using Python.Runtime;

PythonNetLoader.Initialize(typeof(PythonEngine).Assembly, "pyspiel");
$"Runtime Python charge : {PythonEngine.Version}".Display();

public static class OpenSpielBridge
{
    // Map un infoset C# (premier char = nom de carte J/Q/K) vers l'id OpenSpiel
    // (premier char = chiffre de carte 0/1/2, le reste = historique).
    // letter.ToString() est indispensable : Kuhn.CardName est un string[], donc il
    // faut comparer le char comme string ("J"), sinon Array.IndexOf rend -1.
    public static string Id(string csInfoset)
    {
        char letter = csInfoset[0];
        int card = Array.IndexOf(Kuhn.CardName, letter.ToString());
        return ((char)('0' + card)).ToString() + csInfoset.Substring(1);
    }

    // Exploitabilite de la strategie moyenne C# : on la convertit en TabularPolicy
    // Python (dict infoset -> liste de paires (action, proba)) en construisant le
    // dictionnaire du cote Python (builtins dict/list/tuple), sans dependre des API
    // de conversion .NET absentes de pythonnet 3.1.0 ; puis on appelle le vrai
    // exploitability() d'OpenSpiel.
    public static double Exploitability(CFRSolver solver)
    {
        using (Py.GIL())
        {
            dynamic pyspiel = Py.Import("pyspiel");
            dynamic builtins = Py.Import("builtins");
            dynamic game = pyspiel.load_game("kuhn_poker");

            dynamic policyDict = builtins.dict();
            foreach (var iset in Kuhn.AllInfoSets())
            {
                double[] strat = solver.GetAverageStrategy(iset);
                dynamic row = builtins.list();
                for (int a = 0; a < Kuhn.NUM_ACTIONS; a++)
                {
                    dynamic pairL = builtins.list();
                    pairL.append(a);
                    pairL.append(strat[a]);
                    row.append(builtins.tuple(pairL));
                }
                policyDict[Id(iset)] = row;
            }
            dynamic policy = pyspiel.TabularPolicy(policyDict);
            return (double)pyspiel.exploitability(game, policy);
        }
    }
}

// Test de fume : 300 iterations CFR puis exploitabilite reelle via OpenSpiel.
var smoke = new CFRSolver();
for (int t = 0; t < 300; t++) { smoke.Iterations = t + 1; smoke.Cfr("", new[]{0,1}, new double[]{1.0,1.0}); }
$"PONT .NET -> OpenSpiel : OK.   Exploitabilite (CFR, 300 iter) = {OpenSpielBridge.Exploitability(smoke):F6}   (0 = Nash)".Display();

Installing Packages pythonnet

Runtime Python charge : 3.13.15 (tags/v3.13.15:4061bc4, Aug  5 2026, 13:05:39) [MSC v.1944 64 bit (AMD64)]

PONT .NET -> OpenSpiel : OK.   Exploitabilite (CFR, 300 iter) = 0,332812   (0 = Nash)

In [13]:
using System.Diagnostics;

// Benchmark Kuhn Poker : exploitabilite REELLE (OpenSpiel) + temps de calcul.
// Reference : valeur de jeu du joueur 0 = -1/18 ; Nash sur 'Q' = [1.0, 0.0].
const int BUDGET = 1000;
var DONNES = CFRSolver.CardPermutations;   // les 6 donnees

List<(string nom, CFRSolver s)> ConstruireSolveurs() => new()
{
    ("CFR",   new CFRSolver()),
    ("CFR+",  new CFRPlusSolver()),
    ("DCFR",  new DCFRDiscountedSolver()),
    ("OFTRL", new OFTRLSolver()),
};

// "simultaneous" : chaque iteration parcourt les 6 donnees.
// "alternating"  : chaque iteration ne parcourt qu'une donne (rotation) -> moins de mises a jour.
void Entrainer(List<(string, CFRSolver)> solveurs, int iterations, bool alternating)
{
    for (int t = 0; t < iterations; t++)
    {
        var donnees = alternating ? new[] { DONNES[t % DONNES.Length] } : DONNES;
        foreach (var cp in donnees)
            foreach (var (nom, s) in solveurs)
            {
                s.Iterations = t + 1;
                s.Cfr("", cp, new double[] { 1.0, 1.0 });
            }
    }
}

(double secs, List<(string nom, double expl, double[] q)> lignes) Mesurer(int iterations, bool alternating)
{
    var solveurs = ConstruireSolveurs();
    var sw = Stopwatch.StartNew();
    Entrainer(solveurs, iterations, alternating);
    sw.Stop();
    var lignes = new List<(string, double, double[])>();
    foreach (var (nom, s) in solveurs)
        lignes.Add((nom, OpenSpielBridge.Exploitability(s), s.GetAverageStrategy("Q")));
    return (sw.Elapsed.TotalSeconds, lignes);
}

var simultane = Mesurer(BUDGET, alternating: false);
var alterne   = Mesurer(BUDGET, alternating: true);

$"Kuhn poker, {BUDGET} iterations, six donnees.   Reference : valeur jeu J0 = {-1.0/18.0:F6} ; Nash sur 'Q' = [1.0, 0.0]".Display();
Console.WriteLine();
void Imprimer(string mode, double secs, List<(string nom, double expl, double[] q)> lignes)
{
    Console.WriteLine($"{"mode":>13}  {"solveur":>7}  {"exploitabilite":>15}  {"strategie Q":>19}  {"temps total":>12}");
    foreach (var (nom, expl, q) in lignes)
        Console.WriteLine($"{mode,13}  {nom,7}  {expl,15:F6}  {string.Join(",", q.Select(x=>x.ToString("F4"))),19}  {secs,11:F3}s");
    Console.WriteLine($"  {mode,13}  classement (croissant) : " +
        string.Join("  <  ", lignes.OrderBy(l => l.expl).Select(l => $"{l.nom} {l.expl:F4}")) + "\n");
}
Imprimer("simultaneous", simultane.secs, simultane.lignes);
Imprimer("alternating",  alterne.secs,   alterne.lignes);

// Controle de determinisme : les solveurs sont deterministes (aucun tirage aleatoire).
var s1 = ConstruireSolveurs(); Entrainer(s1, 300, false);
var s2 = ConstruireSolveurs(); Entrainer(s2, 300, false);
var id1 = string.Join("|", s1.Select(x => string.Join(",", x.s.GetAverageStrategy("Q").Select(v => v.ToString("F6")))));
var id2 = string.Join("|", s2.Select(x => string.Join(",", x.s.GetAverageStrategy("Q").Select(v => v.ToString("F6")))));
$"Controle de determinisme (300 iter) : strategies identiques = {id1 == id2}".Display();

Kuhn poker, 1000 iterations, six donnees.   Reference : valeur jeu J0 = -0,055556 ; Nash sur 'Q' = [1.0, 0.0]

mode  solveur  exploitabilite  strategie Q  temps total


 simultaneous      CFR         0,006505        0,9988,0,0012        0,122s


 simultaneous     CFR+         0,006784        0,9889,0,0111        0,122s


 simultaneous     DCFR         0,071657        0,6208,0,3792        0,122s


 simultaneous    OFTRL         0,018525        0,9986,0,0014        0,122s


... (4 outputs left out)

  alternating     DCFR         0,073547        0,6184,0,3816        0,014s


  alternating    OFTRL         0,022055        0,9924,0,0076        0,014s


    alternating  classement (croissant) : CFR 0,0185  <  CFR+ 0,0195  <  OFTRL 0,0221  <  DCFR 0,0735



Controle de determinisme (300 iter) : strategies identiques = True

## 4. Lecture du banc

Deux points stables, indépendants de la machine :

1. **Le CFR (vanilla) reste le moins exploitable** du banc, suivi de CFR+ (écrêtage des regrets → léger gain). **L'OFTRL ne bat pas le CFR vanilla sur Kuhn poker** : sur ce jeu très petit, la prédiction optimiste n'a pas de profondeur sur laquelle jouer, et son terme de prédiction ajoute du bruit sans gain mesurable. C'est exactement ce que la théorie annonce : la garantie $O(T^{-3/4})$ de l'OFTRL n'apparaît qu'au-delà d'un seuil d'itérations et sur des jeux où la profondeur porte.
2. **Le DCFR arrive dernier.** Le discount de l'incrément de regret, tel que codé ici, *retarde* la convergence de la somme accumulée : à budget égal, sa stratégie moyenne est la plus éloignée du Nash. Ce n'est pas un bug de l'algorithme — c'est la forme « incrément discounté » que ce notebook expose, et un bon exemple de ce que « la garantie théorique ≠ le classement au banc » veut dire.

> La métrique est l'**exploitabilité réelle** (meilleure réponse OpenSpiel) ; plus elle est basse, plus la stratégie est proche du Nash. Le temps total est un indicateur de coût de calcul à budget d'itérations identique.

> **C.4** — les chiffres du tableau sont produits par la cellule du banc (sortie réelle, re-exécutable) ; l'interprétation ci-dessus les lit, elle ne les invente pas.

## Exercice 1 — Recherche de grille du paramètre OFTRL

Sur le banc du §3, la famille OFTRL est testée avec $(\kappa, \alpha, \beta) = (1.0, 0.5, 1.0)$. **Objectif** : déterminer si un autre point de la grille bat le CFR vanilla.

**Consigne** — compléter `OFTRLGridSearch` : parcourir la grille ci-dessous, pour chaque point mesurer l'exploitabilité (via `OpenSpielBridge.Exploitability`) d'un OFTRL fraîchement entraîné (`iterations` itérations, mode `simultaneous`), et renvoyer le meilleur point (le plus basse exploitabilité) **et** le verdict `batCfr` (le meilleur OFTRL est-il strictement meilleur que le CFR vanilla ?).

> **Indice** : un OFTRL déjà entraîné partage son dictionnaire de regrets — créez **un solveur neuf par point** de la grille, comme le fait `Mesurer`.

In [15]:
// Exercice 1 : recherche de grille du parametre OFTRL.
// Retourner (meilleur, bestExpl, batCfr) : le point (kappa, alpha, beta) le plus exploitable
// (exploitabilite la plus basse) et un bool disant s'il bat strictement le CFR vanilla.
static (OFTRLSolver, double, bool) OFTRLGridSearch()
{
    double[] kappaGrid = { 0.5, 1.0, 2.0, 4.0 };
    double[] alphaGrid = { 0.25, 0.5, 1.0 };
    double[] betaGrid  = { 0.5, 1.0, 2.0 };
    const int iterations = 500;
    var donnees = CFRSolver.CardPermutations;

    OFTRLSolver meilleur = null!;
    double bestExpl = double.MaxValue;
    double cfrVanillaExpl = 0;   // TODO etudiant : mesurer l'exploitabilite du CFR vanilla (reference)

    foreach (var kappa in kappaGrid)
        foreach (var alpha in alphaGrid)
            foreach (var beta in betaGrid)
            {
                // TODO etudiant : creer un OFTRLSolver NEUF, l'entrainer `iterations`
                // iterations sur toutes les `donnees` (mode simultaneous), mesurer son
                // exploitabilite avec OpenSpielBridge.Exploitability, garder le meilleur.
            }

    // TODO etudiant : calculer batCfr = (bestExpl < cfrVanillaExpl).
    return (meilleur, bestExpl, false);   // stub : a completer
}
"Exercice a completer".Display();

Exercice a completer


(9,15): warning CS0219: La variable 'iterations' est assignée, mais sa valeur n'est jamais utilisée

(14,12): warning CS0219: La variable 'cfrVanillaExpl' est assignée, mais sa valeur n'est jamais utilisée



## Exercice 2 — Mode alterné : la convergence se paie-t-elle ?

Le banc du §3 montre deux modes. **Objectif** : quantifier, sur le solveur **CFR vanilla seul**, l'écart d'exploitabilité entre le mode `simultaneous` (6 mises à jour / itération) et le mode `alternating` (1 mise à jour / itération), **à budget d'itérations identique**.

**Consigne** — compléter `ComparaisonModes` : pour `budget` dans `{100, 500, 1000}`, entraîner deux CFR neufs (un par mode) et renvoyer un tableau `(budget, explSimultaneous, explAlternating, ratio)`. Le `ratio` est `explAlternating / explSimultaneous` — combien le mode alterné est-il *pire* à budget égal ?

In [16]:
// Exercice 2 : cout du mode alterne a budget d'iterations identique.
// Retourner une liste de (budget, explSim, explAlt, ratio).
static List<(int budget, double explSim, double explAlt, double ratio)> ComparaisonModes()
{
    var budgets = new[] { 100, 500, 1000 };
    var resultats = new List<(int, double, double, double)>();

    foreach (var budget in budgets)
    {
        // TODO etudiant : entrainer un CFR NEUF en mode simultaneous sur `budget`
        // iterations (toutes les donnees / iteration), mesurer explSim.
        double explSim = 0.0;

        // TODO etudiant : de meme en mode alternating (une donne / iteration, rotation),
        // mesurer explAlt.
        double explAlt = 0.0;

        // TODO etudiant : ratio = explAlt / explSim (garde le diviseur != 0).
        double ratio = 0.0;

        resultats.Add((budget, explSim, explAlt, ratio));
    }
    return resultats;
}
"Exercice a completer".Display();

Exercice a completer

## Exercice 3 — Prédictif OFTRL : deviner le regret de demain

Le cœur de l'OFTRL est la **prédiction** $m_{t+1}$ du regret. **Objectif** : implémenter, sur un mini-jeu à 3 infosets × 4 actions, un prédicteur naïve qui anticipe le regret de l'itération $t+1$ à partir du regret observé à $t$, et mesurer combien cette anticipation *prédit* bien.

**Consigne** — compléter `OFCFPredictor.Predict` (retourner la prédiction du vecteur de regret de la prochaine itération) et `MiniGame3x4` (générer la trajectoire de regrets sur laquelle évaluer). Le score est l'**erreur de prédiction** moyenne : plus elle est basse, plus l'anticipation est fine.

In [17]:
// Exercice 3 : predire l'evolution du regret a partir d'un historique observe.
// Le predicteur renvoie le regret predit pour chaque action du jeu 3x4
// de l'exemple de l'article. Le jeu de test renvoie l'erreur L2 moyenne
// sur le jeu, a comparer aux predicteurs de l'article (cf. reference).
public class OFCFPredictor
{
    // TODO etudiant : remplacer le predicteur trivial (retourne le dernier
    // regret observe) par une prediction fondée sur l'historique.
    // Le plus simple et déjà correct : moyenne mobile des regrets observes
    // par action, ou l'extrapolation lineaire de la derniere tendance.
    public double[] Predict(List<double[]> regretsOberves)
    {
        if (regretsOberves.Count == 0)
            return new double[4];
        return (double[])regretsOberves[^1].Clone();
    }
}

// TODO etudiant : construire le petit jeu 3x4 de l'article, courir le
// predicteur dessus et renvoyer l'erreur L2 moyenne (0 = prediction exacte).
static double MiniGame3x4()
{
    return 0.0;
}
"Exercice a completer".Display();

Exercice a completer

## À retenir

- **Deux familles de CFR « avancés »**, opposées d'esprit : l'**OFTRL** *ajoute* une prédiction du regret (optimiste, garantie $O(T^{-3/4})$ sous calibration) ; le **DCFR** *discoute* l'incrément de regret (décroissant, plus doux).
- **La garantie théorique ne tranche pas le banc.** Sur Kuhn poker, le CFR vanilla reste le moins exploitable et le DCFR le plus : la profondeur sur laquelle l'optimisme joue est ici trop faible pour porter.
- **L'exploitabilité se mesure avec l'outil réel.** Le pont .NET → Python.NET → `pyspiel` → OpenSpiel C++ donne la **meilleure réponse** exacte — c'est cette métrique, pas une distance à l'uniforme, qui départage les solveurs.
- **La parité .NET ⇄ Python est un contrat.** Le jumeau C# définit son socle en place (pas d'`importlib`) et appelle le **même** moteur OpenSpiel que le jumeau Python : les deux carnets restent comparables à la métrique près.

## Vérifiez votre compréhension

1. Pourquoi l'OFTRL **ne bat-il pas** le CFR vanilla sur Kuhn poker alors qu'il a une meilleure garantie théorique ? (indice : la profondeur et le seuil d'itérations)
2. Le DCFR « incrément discounté » de ce notebook **décroît-il** la somme de regrets accumulée ? En quoi se distingue-t-il de la « DCFR à décroissance de la somme » ?
3. Quel est le **rôle exact** du facteur de profondeur $1/(1+\alpha d)$ dans la prédiction OFTRL, et que se passe-t-il quand $\alpha \to 0$ ?
4. Dans le pont Python.NET, pourquoi le mapping infoset C# → id OpenSpiel remplace-t-il la **lettre** de la carte par son **chiffre** (`Q→1`) ?

## Pour aller plus loin

- **Farina, Kroer, Brown, Sandholm (2019)** — *Optimistic Online Learning in Sequential Games*, ICML / PMLR 97 : le cadre stable-prédictif et la garantie $O(T^{-3/4})$.
- **Brown & Sandholm (2019)** — *Solving Imperfect-Information Games via Discounted Regret Minimization*, **Science** 2019 : la référence DCFR, alignée sur le jumeau Python (la motivation du DCFR sur les jeux à profondeur).
- **GT-13e** — [Revision des croyances adversaire](GameTheory-13e-Revision-Croyances-Adversaire-Python.ipynb) : la suite de la série, quand l'adversaire n'est pas le CFR qu'on a supposé.
- **MCCFR** — le jumeau C# du GT-13 : la variante à *matching* (et son jumeau Python), pour comparer un cinquième solveur au banc.

> **Série GameTheory** — [01](GameTheory-01-Setup-Python.ipynb) … [13](GameTheory-13-ImperfectInfo-CFR-CSharp.ipynb) · [13d-Optimistic-CFR](GameTheory-13d-Optimistic-CFR-Python.ipynb) · [13e](GameTheory-13e-Revision-Croyances-Adversaire-Python.ipynb)